# 04 -- TDoA Estimation

Time-Difference-of-Arrival (TDoA) measures the difference in arrival time of a
signal between pairs of geographically separated receivers. The core technique
is **cross-correlation**: by correlating the signals from two receivers we obtain
a function whose peak indicates the relative time delay between them.

PINGU implements several Generalized Cross-Correlation (GCC) weighting schemes
(PHAT, SCOT, ML) that suppress noise and sharpen the correlation peak, together
with sub-sample interpolation methods that refine the peak location to
fractional-sample precision. This notebook walks through each step of the TDoA
estimation pipeline.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt
from scipy.fft import fft, ifft

from pingu.tdoa.gcc import gcc_phat, gcc_scot, gcc_ml, estimate_tdoa
from pingu.tdoa.peak_interpolation import parabolic_interpolation, sinc_interpolation
from pingu.tdoa.pair_manager import PairManager
from pingu.tdoa.uncertainty import crlb_tdoa
from pingu.synthetic.noise import add_awgn

# Reproducible random state
rng = np.random.default_rng(42)

# Common parameters
FS = 48_000.0  # sampling rate (Hz)
N_SAMPLES = 4096  # signal length

print(f"Sampling rate: {FS:.0f} Hz")
print(f"Signal length: {N_SAMPLES} samples ({N_SAMPLES / FS * 1000:.1f} ms)")

## Creating a Delayed Signal Pair

We begin by synthesising a broadband noise signal (bandpass-filtered to a
realistic HF bandwidth) and creating a delayed copy. The delay is applied in
the frequency domain via a linear phase shift, which naturally produces a
fractional (non-integer) sample delay. Additive white Gaussian noise is then
injected at 20 dB SNR to simulate realistic receiver conditions.

In [ ]:
# --- Generate broadband noise signal, bandpass-filtered ---
raw_noise = rng.standard_normal(N_SAMPLES)

# Bandpass filter: 2 kHz -- 8 kHz (a realistic HF sub-band)
f_low, f_high = 2000.0, 8000.0
sos = butter(4, [f_low, f_high], btype="bandpass", fs=FS, output="sos")
clean_signal = sosfilt(sos, raw_noise).astype(np.float64)

# --- Apply a fractional delay of 15.3 samples via FFT phase shift ---
TRUE_DELAY_SAMPLES = 15.3
n_fft = len(clean_signal)
freqs = np.fft.fftfreq(n_fft, d=1.0 / FS)  # Hz
phase_shift = np.exp(-1j * 2.0 * np.pi * freqs * (TRUE_DELAY_SAMPLES / FS))
delayed_signal = np.real(ifft(fft(clean_signal) * phase_shift)).astype(np.float64)

# --- Add AWGN at 20 dB SNR (convert real signals to complex for add_awgn) ---
SNR_DB = 20.0
sig1_complex = clean_signal.astype(np.complex64)
sig2_complex = delayed_signal.astype(np.complex64)

noisy_sig1 = np.real(add_awgn(sig1_complex, SNR_DB, rng=rng).astype(np.complex128))
noisy_sig2 = np.real(add_awgn(sig2_complex, SNR_DB, rng=rng).astype(np.complex128))

# --- Plot both signals overlaid (zoomed to first 200 samples) ---
fig, ax = plt.subplots(figsize=(10, 3))
t_samples = np.arange(200)
ax.plot(t_samples, noisy_sig1[:200], label="Receiver 1 (reference)", alpha=0.8)
ax.plot(t_samples, noisy_sig2[:200], label=f"Receiver 2 (delayed {TRUE_DELAY_SAMPLES} samples)", alpha=0.8)
ax.set_xlabel("Sample index")
ax.set_ylabel("Amplitude")
ax.set_title("Noisy Receiver Signals (first 200 samples)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"True delay: {TRUE_DELAY_SAMPLES} samples = {TRUE_DELAY_SAMPLES / FS * 1e6:.2f} us")
print(f"SNR: {SNR_DB} dB")

## GCC-PHAT Cross-Correlation

GCC-PHAT (Phase Transform) whitens the cross-spectrum magnitude so that only
phase information is used for delay estimation. This produces the sharpest
possible correlation peak, making it the default choice in PINGU.

In [ ]:
# Run GCC-PHAT on the noisy pair
lags_sec, corr_phat = gcc_phat(noisy_sig1, noisy_sig2, fs=FS)

# Locate the peak
peak_idx = np.argmax(corr_phat)
peak_lag_sec = lags_sec[peak_idx]
peak_lag_samples = peak_lag_sec * FS

# Plot the full correlation
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(lags_sec * 1e6, corr_phat, linewidth=0.5)
ax.axvline(peak_lag_sec * 1e6, color="red", linestyle="--", alpha=0.7, label=f"Peak: {peak_lag_sec*1e6:.2f} us")
ax.axvline(TRUE_DELAY_SAMPLES / FS * 1e6, color="green", linestyle=":", alpha=0.7, label=f"True: {TRUE_DELAY_SAMPLES / FS * 1e6:.2f} us")
ax.set_xlabel("Lag (us)")
ax.set_ylabel("GCC-PHAT")
ax.set_title("GCC-PHAT Cross-Correlation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Recovered delay (coarse): {peak_lag_samples:.1f} samples ({peak_lag_sec*1e6:.2f} us)")
print(f"True delay:               {TRUE_DELAY_SAMPLES:.1f} samples ({TRUE_DELAY_SAMPLES / FS * 1e6:.2f} us)")
print(f"Coarse error:             {abs(peak_lag_samples - TRUE_DELAY_SAMPLES):.1f} samples")

## Sub-Sample Interpolation

The coarse peak is limited to integer-sample resolution. Sub-sample
interpolation refines the peak location to fractional precision:

- **Parabolic interpolation** fits a 3-point parabola around the peak -- fast
  and usually sufficient.
- **Sinc interpolation** evaluates the Whittaker-Shannon interpolant on a fine
  grid -- higher accuracy for band-limited signals.

In [ ]:
# --- Parabolic interpolation ---
coarse_idx = int(np.argmax(corr_phat))
para_idx, para_val = parabolic_interpolation(corr_phat, coarse_idx)

# Convert fractional array index back to lag in samples
dt_sec = lags_sec[1] - lags_sec[0]  # seconds per sample in the lag array
para_lag_sec = lags_sec[0] + para_idx * dt_sec
para_lag_samples = para_lag_sec * FS
para_error = abs(para_lag_samples - TRUE_DELAY_SAMPLES)

print("Parabolic interpolation")
print(f"  Coarse peak index:    {coarse_idx}")
print(f"  Interpolated index:   {para_idx:.4f}")
print(f"  Recovered delay:      {para_lag_samples:.4f} samples")
print(f"  Error vs true (15.3): {para_error:.4f} samples")
print()

# --- Sinc interpolation ---
sinc_idx, sinc_val = sinc_interpolation(corr_phat, coarse_idx)
sinc_lag_sec = lags_sec[0] + sinc_idx * dt_sec
sinc_lag_samples = sinc_lag_sec * FS
sinc_error = abs(sinc_lag_samples - TRUE_DELAY_SAMPLES)

print("Sinc interpolation")
print(f"  Interpolated index:   {sinc_idx:.4f}")
print(f"  Recovered delay:      {sinc_lag_samples:.4f} samples")
print(f"  Error vs true (15.3): {sinc_error:.4f} samples")
print()

# --- Comparison table ---
print(f"{'Method':<22} {'Delay (samples)':>16} {'Error (samples)':>16}")
print("-" * 56)
print(f"{'Coarse (integer)':<22} {coarse_idx * dt_sec * FS:>16.4f} {abs(coarse_idx * dt_sec * FS - TRUE_DELAY_SAMPLES):>16.4f}")
print(f"{'Parabolic':<22} {para_lag_samples:>16.4f} {para_error:>16.4f}")
print(f"{'Sinc':<22} {sinc_lag_samples:>16.4f} {sinc_error:>16.4f}")

## Comparing GCC Methods

PINGU provides three GCC weighting schemes:

| Method | Weighting | Characteristics |
|--------|-----------|----------------|
| PHAT   | `1 / abs(G_xy)` | Sharpest peak; robust in multi-path |
| SCOT   | `1 / sqrt(G_xx * G_yy)` | Good balance of peak sharpness and noise robustness |
| ML     | Maximum-likelihood | Optimal under Gaussian noise model |

In [ ]:
# Run all three GCC methods on the same signal pair
lags_phat, corr_phat_cmp = gcc_phat(noisy_sig1, noisy_sig2, fs=FS)
lags_scot, corr_scot_cmp = gcc_scot(noisy_sig1, noisy_sig2, fs=FS)
lags_ml, corr_ml_cmp = gcc_ml(noisy_sig1, noisy_sig2, fs=FS)

# Zoom to region around the true delay for clearer comparison
true_delay_sec = TRUE_DELAY_SAMPLES / FS
zoom_width = 100.0 / FS  # +/- 100 samples around the peak

fig, ax = plt.subplots(figsize=(10, 4))

for lags, corr, label, color in [
    (lags_phat, corr_phat_cmp, "GCC-PHAT", "tab:blue"),
    (lags_scot, corr_scot_cmp, "GCC-SCOT", "tab:orange"),
    (lags_ml, corr_ml_cmp, "GCC-ML", "tab:green"),
]:
    mask = np.abs(lags - true_delay_sec) < zoom_width
    # Normalise each correlation to unit peak for visual comparison
    peak_val = np.max(np.abs(corr))
    normed = corr / peak_val if peak_val > 0 else corr
    ax.plot(lags[mask] * 1e6, normed[mask], label=label, color=color, alpha=0.85)

ax.axvline(true_delay_sec * 1e6, color="red", linestyle="--", alpha=0.6, label="True delay")
ax.set_xlabel("Lag (us)")
ax.set_ylabel("Normalised correlation")
ax.set_title("Comparison of GCC Methods (zoomed around true delay)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print recovered delays
print(f"{'Method':<12} {'Peak lag (samples)':>20} {'Error (samples)':>18}")
print("-" * 52)
for name, lags, corr in [
    ("PHAT", lags_phat, corr_phat_cmp),
    ("SCOT", lags_scot, corr_scot_cmp),
    ("ML", lags_ml, corr_ml_cmp),
]:
    pk = np.argmax(corr)
    pk_frac, _ = parabolic_interpolation(corr, pk)
    dt = lags[1] - lags[0]
    recovered = (lags[0] + pk_frac * dt) * FS
    print(f"{name:<12} {recovered:>20.4f} {abs(recovered - TRUE_DELAY_SAMPLES):>18.4f}")

print("\nPHAT produces the sharpest peak; SCOT and ML have broader side-lobes.")

## Effect of SNR on Accuracy

As SNR decreases the cross-correlation peak becomes noisier, and the delay
estimate degrades. The Cramer-Rao Lower Bound (CRLB) provides a theoretical
floor on the achievable variance for any unbiased estimator.

In [ ]:
SNR_VALUES = [30, 20, 10, 5, 0, -5]  # dB
N_TRIALS = 50  # average over multiple noise realisations
BANDWIDTH = f_high - f_low  # Hz
INTEGRATION_TIME = N_SAMPLES / FS  # seconds

mean_errors = []   # mean absolute delay error in seconds
crlb_values = []   # CRLB standard deviation in seconds

for snr_db in SNR_VALUES:
    snr_linear = 10.0 ** (snr_db / 10.0)
    trial_errors = []

    for trial in range(N_TRIALS):
        trial_rng = np.random.default_rng(1000 + trial)
        # Create fresh noisy copies at this SNR
        n1 = np.real(add_awgn(sig1_complex, snr_db, rng=trial_rng).astype(np.complex128))
        n2 = np.real(add_awgn(sig2_complex, snr_db, rng=trial_rng).astype(np.complex128))

        lags_s, cc = gcc_phat(n1, n2, fs=FS)
        pk = int(np.argmax(cc))
        # Sub-sample refinement
        if 0 < pk < len(cc) - 1:
            frac_idx, _ = parabolic_interpolation(cc, pk)
        else:
            frac_idx = float(pk)
        dt = lags_s[1] - lags_s[0]
        recovered_sec = lags_s[0] + frac_idx * dt
        true_sec = TRUE_DELAY_SAMPLES / FS
        trial_errors.append(abs(recovered_sec - true_sec))

    mean_errors.append(np.mean(trial_errors))

    # CRLB (square root to get std dev)
    if snr_linear > 0:
        crlb_var = crlb_tdoa(
            bandwidth=BANDWIDTH,
            snr_linear=snr_linear,
            integration_time=INTEGRATION_TIME,
        )
        crlb_values.append(np.sqrt(crlb_var))
    else:
        crlb_values.append(np.nan)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(SNR_VALUES, np.array(mean_errors) * 1e6, "o-", label="GCC-PHAT mean |error|")
ax.semilogy(SNR_VALUES, np.array(crlb_values) * 1e6, "s--", color="red", label="CRLB (std dev)")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delay error / bound (us)")
ax.set_title("TDoA Estimation Accuracy vs SNR")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"{'SNR (dB)':>10} {'Mean error (us)':>16} {'CRLB std (us)':>16}")
print("-" * 44)
for snr, err, crlb_std in zip(SNR_VALUES, mean_errors, crlb_values):
    print(f"{snr:>10} {err * 1e6:>16.4f} {crlb_std * 1e6:>16.4f}")

## Pair Manager -- All 10 Receiver Pairs

With 5 receivers there are C(5, 2) = 10 unique pairs. The `PairManager` class
enumerates them and orchestrates TDoA estimation across all pairs in one call.

In [ ]:
# Create a PairManager with 5 receivers
receiver_ids = ["RX0", "RX1", "RX2", "RX3", "RX4"]
pm = PairManager(receiver_ids)

print(f"Number of receivers: {len(receiver_ids)}")
print(f"Number of unique pairs: {pm.n_pairs}")
print("\nAll pairs:")
for i, pair in enumerate(pm.get_pairs()):
    print(f"  {i:>2d}: {pair[0]} -- {pair[1]}")

# --- Simulate 5 receivers observing the same broadband source ---
# Define per-receiver delays (samples) based on a simple geometry.
# A source at some location arrives at each receiver with a different propagation delay.
true_delays_samples = np.array([0.0, 5.7, -3.2, 12.1, 8.4])  # relative to RX0

signals = {}
sim_rng = np.random.default_rng(123)
for rx_id, delay_samp in zip(receiver_ids, true_delays_samples):
    # Apply fractional delay via FFT phase shift
    phase = np.exp(-1j * 2.0 * np.pi * freqs * (delay_samp / FS))
    delayed = np.real(ifft(fft(clean_signal) * phase)).astype(np.float64)
    # Add noise
    noisy = np.real(add_awgn(delayed.astype(np.complex64), 20.0, rng=sim_rng).astype(np.complex128))
    signals[rx_id] = noisy

# Estimate all 10 TDoAs
estimates = pm.estimate_all_tdoas(signals, fs=FS, method="phat")

print(f"\n{'Pair':<14} {'Est. delay (us)':>16} {'True diff (us)':>16} {'Error (us)':>12}")
print("-" * 60)
for est in estimates:
    i_idx = receiver_ids.index(est.receiver_i)
    j_idx = receiver_ids.index(est.receiver_j)
    true_diff_sec = (true_delays_samples[i_idx] - true_delays_samples[j_idx]) / FS
    error_us = (est.delay - true_diff_sec) * 1e6
    print(
        f"{est.receiver_i}-{est.receiver_j:<8} "
        f"{est.delay * 1e6:>16.2f} "
        f"{true_diff_sec * 1e6:>16.2f} "
        f"{error_us:>12.2f}"
    )

## Summary

- **GCC-PHAT** is the default TDoA estimator in PINGU. Its phase-transform
  weighting produces the sharpest correlation peak, making it robust in
  moderate-to-high SNR conditions.
- **Sub-sample interpolation** (parabolic or sinc) refines the coarse
  integer-sample peak to fractional precision, typically achieving errors
  well within 0.1 samples at SNRs above 10 dB.
- The **CRLB** provides a theoretical lower bound on TDoA variance that
  depends on bandwidth, SNR, and integration time. It serves as the
  benchmark against which estimator performance is evaluated.
- The **PairManager** automates pair enumeration and batch TDoA estimation
  across all C(N, 2) receiver pairs.